In [1]:
# %pip install loguru

In [2]:
import sys
from loguru import logger
from typing import Callable
from functools import wraps
from string import Template
import traceback


In [3]:
logger.remove()

logger.add(
    sys.stdout,
    colorize=True,
    format="<level>[{level}]</level>[<green>{time:YYYY-MM-DD HH:mm:ss}</green>][<cyan>{name}:{function}:{line}</cyan>] <level>{message}</level>",
    level="TRACE" # default is DEBUG
)

# logger.add(
#     "log.log",
#     colorize=False,
#     format="[{level}][{time:YYYY-MM-DD HH:mm:ss}][{name}:{function}:{line}] {message}"
# )

logger.trace(0)
logger.debug(0)
logger.info(1)
logger.error(2)
logger.warning(3)
logger.critical(4)
logger.success(5)
logger.exception(6)
logger.log(7, "test")

[TRACE][2026-02-15 03:46:14][__main__:<module>:16] 0
[DEBUG][2026-02-15 03:46:14][__main__:<module>:17] 0
[INFO][2026-02-15 03:46:14][__main__:<module>:18] 1
[ERROR][2026-02-15 03:46:14][__main__:<module>:19] 2
[DEBUG][2026-02-15 03:46:14][__main__:<module>:17] 0
[INFO][2026-02-15 03:46:14][__main__:<module>:18] 1
[ERROR][2026-02-15 03:46:14][__main__:<module>:19] 2
[WARNING][2026-02-15 03:46:14][__main__:<module>:20] 3
[CRITICAL][2026-02-15 03:46:14][__main__:<module>:21] 4
[SUCCESS][2026-02-15 03:46:14][__main__:<module>:22] 5
[ERROR][2026-02-15 03:46:14][__main__:<module>:23] 6
NoneType: None
[Level 7][2026-02-15 03:46:14][__main__:<module>:24] test


In [4]:
error_template = Template("""\x1b[0m\x1b[31m[${funct_name}] has error:
${error}\x1b[0m""")

In [5]:
def logger_wrapper(func: Callable) -> Callable:
    """
    A decorator wraps a function and provide error logging.
    If the wrap function raises an exception, the decorator will log the error message through logger

    Args:
        func (Callable): the function to be wrapped.

    Returns:
        Callable: the wrapped function with error logging
    """

    @wraps(func)
    def wrap(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            tb = "".join(traceback.TracebackException.from_exception(e).format())
            logger.error(error_template.safe_substitute(funct_name=func.__name__, error=tb))

    return wrap

In [7]:
@logger_wrapper
def func_1():
    logger.info(0/0)
    logger.error(1)

func_1()

[ERROR][2026-02-15 03:46:24][__main__:wrap:19] [func_1] has error:
Traceback (most recent call last):
  File "/tmp/ipykernel_340/1729173488.py", line 16, in wrap
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_340/322063522.py", line 3, in func_1
    logger.info(0/0)
                ~^~
ZeroDivisionError: division by zero

